In [ ]:
import math
import sys
from pathlib import Path

from datasets import DatasetDict, concatenate_datasets, load_dataset
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, "..")

from scripts.active_learner import ActiveLearning
from scripts.dataloader.offline import OffinePairedDataLoader
from scripts.pairmodels import PairComparisonBTModel, PairComparisonLogisticModel, PairComparisonModelBase
from scripts.policies import KroneckerRankingPolicy
from scripts.sampling import AdaptiveSamplingPolicy, RoundRobinSamplingPolicy

CACHE_DIR = "../data/raw"
FIGURE_DIR = Path("../data/figures")

In [ ]:
def perform_real_data_study(
    dataset: str | DatasetDict,
    n_arms: int,
    context_dim: int,
    n_rounds: int,
    model_type: str = "bt_model",
    policy_class: str = "kronecker_ranking",
    sampling_strategy: str = "adaptive",
    correct_for_bias: bool = True,
    context_key: str = "context",
    model_a_key: str = "model_a",
    model_b_key: str = "model_b",
    response_key: str = "Y",
    seed: int = 1234,
):
    dataloader = OffinePairedDataLoader(n_arms, seed=seed)
    dataloader.load_arena_data(
        dataset, context_key=context_key, model_a_key=model_a_key, model_b_key=model_b_key, response_key=response_key
    )

    if model_type == "bt_model":
        pair_model = PairComparisonBTModel(n_arms, context_dim)
    elif model_type == "logistic":
        pair_model = PairComparisonLogisticModel(n_arms, context_dim)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    if policy_class == "kronecker_ranking":
        reco_policy = KroneckerRankingPolicy(n_arms, context_dim)
    else:
        raise ValueError(f"Unknown policy_class: {policy_class}")

    if sampling_strategy == "adaptive":
        sampling_policy = AdaptiveSamplingPolicy(n_arms, context_dim)
    elif sampling_strategy == "uniform":
        sampling_policy = RoundRobinSamplingPolicy(n_arms, context_dim)
    else:
        raise ValueError(f"Unknown sampling_strategy: {sampling_strategy}")

    # Allow immediate adaptive behavior
    sampling_policy.initial_round = 0

    learner = ActiveLearning(
        data=dataloader, pair_model=pair_model, sampling_policy=sampling_policy, reco_policy=reco_policy
    )

    data_history, final_policy = learner.perform_rounds(
        n_rounds,
        seed=seed,
        correct_for_bias=correct_for_bias,
        verbose=True,
        measure_func=dataloader.calculate_performance_metrics,
    )
    return pd.DataFrame(data_history), final_policy, pair_model


def compare_uniform_and_adaptive_sampling(
    dataset,
    n_rounds: int,
    n_arms: int,
    context_dim: int,
    model_type: str = "bt_model",
    context_key: str = "context",
):
    df_uni, pi_uni, _ = perform_real_data_study(
        dataset, n_arms, context_dim, n_rounds,
        model_type=model_type, sampling_strategy="uniform",
        context_key=context_key
    )
    df_ada, pi_ada, pm_ada = perform_real_data_study(
        dataset, n_arms, context_dim, n_rounds,
        model_type=model_type, sampling_strategy="adaptive",
        context_key=context_key
    )
    df_uni["sampling"] = "uniform"
    df_ada["sampling"] = "adaptive"
    return pd.concat([df_uni, df_ada]), pi_ada, pm_ada

def make_comparison_plot(df: pd.DataFrame, dataset_name: str, save_plot=True, ylim=None):
    n_rounds = len(df) // 2
    
    unif_train = df[df['sampling'] == 'uniform']['policy_value'].apply(lambda x: x[0]).tolist()
    unif_test = df[df['sampling'] == 'uniform']['policy_value'].apply(lambda x: x[1]).tolist()
    ada_train = df[df['sampling'] == 'adaptive']['policy_value'].apply(lambda x: x[0]).tolist()
    ada_test = df[df['sampling'] == 'adaptive']['policy_value'].apply(lambda x: x[1]).tolist()
    
    plot_df = pd.DataFrame({
        "round": list(range(n_rounds)) * 4,
        "method": ["Uniform"] * (2 * n_rounds) + ["Adaptive"] * (2 * n_rounds),
        "regime": (["Train"] * n_rounds + ["Test"] * n_rounds) * 2,
        "accuracy": unif_train + unif_test + ada_train + ada_test
    })

    plt.figure(figsize=(6, 4.5))
    sns.set_theme(style="white", context="paper", font_scale=1.3)

    ax = sns.lineplot(
        data=plot_df,
        x="round",
        y="accuracy",
        hue="method",
        style="regime",
        linewidth=2.2,
        dashes={
            "Train": "",
            "Test": (1, 1)
        }
    )

    handles = [
        Line2D([0], [0], color=sns.color_palette()[0], lw=2.2, linestyle='-', label="Uniform, Train"),
        Line2D([0], [0], color=sns.color_palette()[0], lw=2.2, linestyle=':', label="Uniform, Test"),
        Line2D([0], [0], color=sns.color_palette()[1], lw=2.2, linestyle='-', label="Adaptive, Train"),
        Line2D([0], [0], color=sns.color_palette()[1], lw=2.2, linestyle=':', label="Adaptive, Test"),
    ]

    ax.legend(
        handles=handles,
        title="Sampling Method x Regime",
        loc="best",
        frameon=True
    )
    ax.set_xlabel("Round")
    ax.set_ylabel("Accuracy")
    if ylim is not None:
        ax.set_ylim(ylim[0], ylim[1])

    sns.despine()
    plt.tight_layout()

    if save_plot:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        figpath = FIGURE_DIR / f"llm_{dataset_name}.pdf"
        plt.savefig(figpath, dpi=300, bbox_inches="tight", metadata={"CreationDate": None})
        print(f"Saved plot to {figpath}")

    plt.show()

In [ ]:
# build theta plot
def build_theta(
    dataset: str | DatasetDict,
    n_arms: int,
    pair_model: PairComparisonModelBase,
    context_categories: list,
    context_key: str = "context",
    model_a_key: str = "model_a",
    model_b_key: str = "model_b",
    response_key: str = "Y",
):
    # create dataset copy
    dataloader = OffinePairedDataLoader(n_arms = n_arms, seed=1234)
    dataloader.load_arena_data(
        dataset, context_key=context_key, model_a_key=model_a_key, 
        model_b_key=model_b_key, response_key=response_key
    )

    theta_df = []
    for i, cat in enumerate(context_categories):
        context_vec = np.zeros(len(context_categories))
        context_vec[i] = 1
        theta = pair_model.get_estimated_theta(context_vec)
        theta_df.append(pd.DataFrame({
            "category": cat,
            "llm": list(dataloader.model_index_mapping.keys()),
            "theta": [theta[dataloader.model_index_mapping[m]] for m in dataloader.model_index_mapping]
        }))

    theta_df = pd.concat(theta_df)
    return theta_df, dataloader

def build_theta_plot(
    theta_df: pd.DataFrame,
    dataloader: OffinePairedDataLoader,
    category_list: list,
    dataset_name: str,
    save_plot: bool = True
):
    base_model = dataloader.MODEL_LIST[0]
    df_plot = theta_df[theta_df["llm"] != base_model].copy()

    # Number of facets
    n_cat = len(category_list)

    # Global symmetric x-limit
    max_abs = df_plot["theta"].abs().max()
    x_limit = max_abs * 1.1

    # Grid size (compact layout)
    ncols = 4   # change if you want wider grid
    nrows = math.ceil(n_cat / ncols)
    sns.set_theme(style="white", context="paper", font_scale=1.3)
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(16, 2 * nrows),
        sharex=True
    )

    axes = axes.flatten()

    for i, cat in enumerate(category_list):
        ax = axes[i]
        subset = df_plot[df_plot["category"] == cat].copy()
        
        # Sort by theta for clean visual ordering
        subset = subset.sort_values("llm")
        y_positions = np.arange(len(subset))
        
        # Baseline
        ax.axvline(0, linestyle="--", linewidth=1, color = "black")
        
        # Draw sticks and points
        for y, theta in zip(y_positions, subset["theta"]):
            color = "blue" if theta > 0 else "red"
            
            ax.hlines(y=y, xmin=0, xmax=theta, linewidth=2, color=color)
            ax.scatter(theta, y, s=40, color=color, zorder=3)
        
        ax.set_yticks(y_positions)
        ax.set_yticklabels(subset["llm"], fontsize = 12)
        ax.set_title(cat, fontsize=14)
        ax.set_xlim(-x_limit, x_limit)

    # Remove empty subplots if any
    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    sns.despine()
    plt.tight_layout()

    # save the figure
    if save_plot:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        figpath = FIGURE_DIR / f"llm_{dataset_name}_theta.pdf"
        plt.savefig(figpath, dpi=300, bbox_inches="tight", metadata={"CreationDate": None})
        print(f"Plot saved in {figpath}")

    plt.show()


# MT Bench Dataset

In [ ]:
CATEGORIES = ["Writing", "Roleplay", "Reasoning", "Math", "Coding", "Extraction", "STEM", "Humanities"]

def clean_row_mtbench(row):
    category = (row['question_id'] - 81) // 10 
    one_hot_cat = np.zeros(len(CATEGORIES))
    one_hot_cat[category] = 1.0

    conv_list = row['conversation_a']
    question = []
    for conv in conv_list:
        if conv['role'] == "user":
            question.append(conv['content'])

    return {
        "question": " ".join(question),
        "model_a": row['model_a'],
        "model_b": row['model_b'],
        "Y": 1 if row['winner'] == 'model_a' else 0,
        "category": category,
        "onehot_category": one_hot_cat
    }

ds_mt = load_dataset("lmsys/mt_bench_human_judgments", cache_dir=CACHE_DIR)
human_ds = ds_mt['human'].filter(lambda row: row['winner'] in ['model_a', 'model_b'])
gpt_ds = ds_mt['gpt4_pair'].filter(lambda row: row['winner'] in ['model_a', 'model_b'])
human_processed = human_ds.map(clean_row_mtbench, remove_columns=human_ds.column_names)
gpt_processed = gpt_ds.map(clean_row_mtbench, remove_columns=gpt_ds.column_names)
processed_mt_ds = concatenate_datasets([human_processed, gpt_processed])
split_ds = processed_mt_ds.train_test_split(test_size=0.25, seed=1234, shuffle=True)
mt_bench_ds = DatasetDict({"train": split_ds['train'], "test": split_ds['test']})
mt_bench_ds

In [ ]:
df_mt, final_policy_mt, pair_model_mt = compare_uniform_and_adaptive_sampling(
    mt_bench_ds,
    n_rounds=100,
    n_arms=6,
    context_dim=len(CATEGORIES),
    model_type="bt_model",
    context_key="onehot_category"
)

make_comparison_plot(df_mt, "mt_bench", save_plot=True)

In [ ]:
theta_df, dataloader = build_theta(
    mt_bench_ds, n_arms=6, pair_model=pair_model_mt,
    context_categories=CATEGORIES, context_key="onehot_category"
)
theta_df.head()

In [ ]:
build_theta_plot(theta_df, dataloader, CATEGORIES, "mt_bench", save_plot=True)

In [ ]:
# train the SVM classifier

# tf-idf + SVM pipeline to predict categories
X_train = []
y_train = []

for key in ['train', 'test']:
    for row in mt_bench_ds[key]:
        X_train.append(row['question'])
        y_train.append(row['category'])

print(len(X_train), len(y_train))

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),   # unigrams + bigrams
        stop_words="english"
    )),
    ("clf", LinearSVC())
])
pipeline.fit(X_train, y_train)

In [ ]:
CATEGORIES[pipeline.predict(["What is 2 + 2?"])[0]]

## Chatbot Arena Dataset

In [ ]:
# load dataset and clean

cb_ds = load_dataset("lmsys/chatbot_arena_conversations", cache_dir=CACHE_DIR)

def clean_row(row):
    conv_list = row['conversation_a']
    question = []
    for conv in conv_list:
        if conv['role'] == "user":
            question.append(conv['content'])

    return {
        "question": " ".join(question),
        "model_a": row['model_a'],
        "model_b": row['model_b'],
        "Y": 1 if row['winner'] == 'model_a' else 0
    }

filtered_ds = cb_ds['train']
processed_ds = filtered_ds.map(clean_row, remove_columns=filtered_ds.column_names)

# predict the question categories
X_test = [row['question'] for row in processed_ds]
y_pred = pipeline.predict(X_test)
y_onehot_pred = []
for y in y_pred:
    one_hot_cat = np.zeros(len(CATEGORIES))
    one_hot_cat[y] = 1
    y_onehot_pred.append(one_hot_cat)
    
processed_ds = processed_ds.add_column("category", y_pred)
processed_ds = processed_ds.add_column("onehot_category", y_onehot_pred)

split_ds = processed_ds.train_test_split(test_size=0.25, seed=1234, shuffle=True)
train_ds = split_ds['train']
test_ds = split_ds['test']


cb_arena_ds = DatasetDict({
    "train": train_ds,
    "test": test_ds
})
cb_arena_ds

In [ ]:
# total number of models
model_list = set([r['model_a'] for r in cb_arena_ds['train']] + [r['model_b'] for r in cb_arena_ds['train']])
print(f'There are {len(model_list)} models')
model_list

In [ ]:
df_cb, final_policy_cb, pair_model_cb = compare_uniform_and_adaptive_sampling(
    cb_arena_ds,
    n_rounds=100,
    n_arms=len(model_list),
    context_dim=len(CATEGORIES),
    model_type="bt_model",
    context_key="onehot_category"
)

make_comparison_plot(df_cb, "cb_arena", save_plot=True)

In [ ]:
theta_df, dataloader = build_theta(
    cb_arena_ds, n_arms=len(model_list), pair_model=pair_model_cb,
    context_categories=CATEGORIES, context_key="onehot_category"
)
print(theta_df.shape)
theta_df.head()

In [ ]:
theta_subdf = theta_df.loc[theta_df['llm'].isin([
    'alpaca-13b', 'claude-v1', 'gpt-3.5-turbo', 'gpt-4', 'llama-13b'
])]
build_theta_plot(theta_subdf, dataloader, CATEGORIES, "cb_arena", save_plot=True)

# LMArena 140k Dataset

In [ ]:
arena_ds = load_dataset("lmarena-ai/arena-human-preference-140k", cache_dir=CACHE_DIR)
arena_ds

In [ ]:
# as there are multiple models, we can get the most popular models
from collections import Counter
import pandas as pd

train_ds = arena_ds['train']
total_counts = Counter(train_ds["model_a"]) + Counter(train_ds["model_b"])
top_models = total_counts.most_common(20)  # Top models by appearance
top_models

In [ ]:
selected_models = ["claude-opus-4-20250514", "gemini-2.5-flash", "mistral-medium-2505", "qwen3-235b-a22b-no-thinking", "o3-2025-04-16"]

def clean_row(row):
    def get_assistant_text(conv):
        for turn in conv:
            if turn["role"] == "assistant" and len(turn['content']) > 0:
                return turn["content"][0].get("text", "")

    def get_user_prompt(conv):
        for turn in conv:
            if turn["role"] == "user" and len(turn['content']) > 0:
                return turn["content"][0].get("text", "")

    return {
        "model_a": row['model_a'],
        "model_b": row['model_b'],
        "question": get_user_prompt(row['conversation_a']),
        "tags": {
            "creative_writing": row['category_tag']['creative_writing_v0.1']['creative_writing'],
            "creativity": row['category_tag']['criteria_v0.1']['creativity'],
            "domain_knowledge": row['category_tag']['criteria_v0.1']['domain_knowledge'],
            "problem_solving": row['category_tag']['criteria_v0.1']['problem_solving'],
            "real_world": row['category_tag']['criteria_v0.1']['real_world'],
            "specificity": row['category_tag']['criteria_v0.1']['specificity'],
            "technical_accuracy": row['category_tag']['criteria_v0.1']['technical_accuracy'],
            "math": row['category_tag']['math_v0.1']['math'],
            "code": row['is_code']
        },
        "response_a": get_assistant_text(row['conversation_a']),
        "response_b": get_assistant_text(row['conversation_b']),
        "Y": int(row["winner"] == "model_a")
    }

train_ds = arena_ds['train']
filtered_ds = train_ds.filter(lambda row: row["model_a"] in selected_models and row["model_b"] in selected_models)
processed_ds = filtered_ds.map(clean_row, remove_columns=filtered_ds.column_names)

X_test = [row['question'] for row in processed_ds]
y_pred = pipeline.predict(X_test)

y_onehot_pred = []
for y in y_pred:
    one_hot_cat = np.zeros(len(CATEGORIES))
    one_hot_cat[y] = 1
    y_onehot_pred.append(one_hot_cat)
    
processed_ds = processed_ds.add_column("category", y_pred)
processed_ds = processed_ds.add_column("onehot_category", y_onehot_pred)
processed_ds

In [ ]:
split_ds = processed_ds.train_test_split(test_size=0.25, seed=1234)
train_ds = split_ds['train']
test_ds = split_ds['test']

lmarena_ds = DatasetDict({
    "train": train_ds,
    "test": test_ds
})
lmarena_ds

In [ ]:
df_arena, final_policy_arena, pair_model_arena = compare_uniform_and_adaptive_sampling(
    lmarena_ds,
    n_rounds=300,
    n_arms=len(selected_models),
    context_dim=len(CATEGORIES),
    model_type="bt_model",
    context_key="onehot_category"
)
make_comparison_plot(df_arena, "lm_arena", save_plot=True)

In [ ]:
theta_df, dataloader = build_theta(
    lmarena_ds, n_arms=len(selected_models), pair_model=pair_model_arena,
    context_categories=CATEGORIES, context_key="onehot_category"
)
print(theta_df.shape)
theta_df.head()

In [ ]:
build_theta_plot(theta_df, dataloader, CATEGORIES, "lm_arena", save_plot=True)